In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
import joblib

In [2]:
df = pd.read_csv('final_internship_data.csv')
df.shape

(3073, 26)

In [3]:
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates()

df = df.dropna(how='all', subset=[c for c in df.columns if c not in ['User ID','User Name','Driver Name']])
df = df.drop(columns=['User ID', 'User Name', 'Driver Name', 'key'])
df.shape

Duplicate rows: 0


(3072, 22)

In [4]:
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df = df.drop(columns=['pickup_datetime', 'hour'])

df = df[df['fare_amount'] > 0]
df = df[df['passenger_count'] > 0]
df = df[df['distance'] > 0]
df = df[df['distance'] <= 100]
df.shape

(2975, 22)

In [5]:
for col in ['fare_amount', 'distance']:
    cap_value = df[col].quantile(0.99)
    df[col] = np.where(df[col] > cap_value, cap_value, df[col])

df[['fare_amount', 'distance']].describe()

,fare_amount,distance
count,2975.000000,2975.000000
mean,11.221277,3.341572
std,8.717349,3.460468
min,0.010000,0.000279
25%,6.000000,1.273928
50%,8.500000,2.188903
75%,12.900000,4.032141
max,52.000000,20.233018


In [9]:
distance_cap = df['distance'].quantile(0.99) 
fare_cap = df['fare_amount'].quantile(0.99) 

In [8]:
df = df.drop(columns=['ewr_dist', 'lga_dist', 'sol_dist', 'nyc_dist'])
df.shape

(2975, 18)

In [10]:
X = df.drop(columns=['fare_amount'])
y = df['fare_amount']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

X_train.shape, X_test.shape

((2380, 17), (595, 17))

In [11]:
car_condition_order = ['Bad', 'Good', 'Very Good', 'Excellent']
traffic_condition_order = ['Flow Traffic', 'Dense Traffic', 'Congested Traffic']

ordinal_cols = ['Car Condition', 'Traffic Condition']
onehot_cols = ['Weather']
numeric_cols = [c for c in X.columns if c not in ordinal_cols + onehot_cols]

preprocessor = ColumnTransformer(transformers=[
    ('ordinal_car', OrdinalEncoder(categories=[car_condition_order]), ['Car Condition']),
    ('ordinal_traffic', OrdinalEncoder(categories=[traffic_condition_order]), ['Traffic Condition']),
    ('onehot', OneHotEncoder(handle_unknown='ignore'), onehot_cols),
    ('scale', StandardScaler(), numeric_cols),
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

X_train_processed.shape, X_test_processed.shape

((2380, 21), (595, 21))

In [12]:
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(
        n_estimators=200, min_samples_split=2, min_samples_leaf=1,
        max_features='log2', max_depth=None, random_state=42
    ),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42)
}

results = []

for name, model in models.items():
    model.fit(X_train_processed, y_train_log)
    y_pred_log = model.predict(X_test_processed)
    y_pred = np.expm1(y_pred_log)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    results.append({'Model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2})
    print(f"{name}: MAE={mae:.4f}, RMSE={rmse:.4f}, R2={r2:.4f}")

results_df = pd.DataFrame(results).sort_values('RMSE')
results_df

Linear Regression: MAE=2.9639, RMSE=6.7299, R2=0.4367
Decision Tree: MAE=2.6571, RMSE=4.7489, R2=0.7195
Random Forest: MAE=2.2332, RMSE=4.2326, R2=0.7772
Gradient Boosting: MAE=1.9176, RMSE=3.7890, R2=0.8214


,Model,MAE,RMSE,R2
3,Gradient Boosting,1.917594,3.789046,0.821436
2,Random Forest,2.233159,4.232638,0.777179
1,Decision Tree,2.657059,4.748936,0.719504
0,Linear Regression,2.963903,6.729879,0.436688


In [13]:
import time

for name, model in models.items():
    start = time.time()
    _ = model.predict(X_test_processed)
    elapsed = time.time() - start
    print(f"{name}: prediction time for {X_test_processed.shape[0]} rows = {elapsed*1000:.2f} ms")

Linear Regression: prediction time for 595 rows = 0.36 ms
Decision Tree: prediction time for 595 rows = 0.53 ms
Random Forest: prediction time for 595 rows = 30.07 ms
Gradient Boosting: prediction time for 595 rows = 1.08 ms


## Model Comparison & Justification

| Model | MAE | RMSE | R² |
|---|---|---|---|
| Gradient Boosting | 1.918 | 3.789 | 0.821 |
| Random Forest | 2.233 | 4.233 | 0.777 |
| Decision Tree | 2.657 | 4.749 | 0.720 |
| Linear Regression | 2.964 | 6.730 | 0.437 |

**Chosen model: Gradient Boosting Regressor**

We selected Gradient Boosting as the final model because it outperformed all other models on every test-set metric: it achieved the lowest MAE ($1.92 vs. $2.23 for the next-best model, Random Forest), the lowest RMSE (3.79 vs. 4.23), and the highest R² (0.82, meaning it explains 82% of the variance in fare amount vs. 78% for Random Forest). The gap over Linear Regression is especially large (R² of 0.44 vs. 0.82), confirming that the relationship between trip features and fare is not purely linear and that tree-based ensemble methods capture it far better. Beyond accuracy, Gradient Boosting is also efficient at inference time (1.08ms for 595 rows), roughly 28x faster than Random Forest (30ms), which matters for a real-time Flask prediction service. While it is less directly interpretable than Linear Regression or a single Decision Tree, the substantial accuracy gain justifies this trade-off for a fare-prediction use case where correctness matters more than transparency. For these reasons — best accuracy across all three metrics, fast prediction speed, and evidence-based (not training-error-only) selection — Gradient Boosting is the model we deploy in Part B and C.

In [14]:
final_model = GradientBoostingRegressor(random_state=42)
final_model.fit(X_train_processed, y_train_log)
print("Model trained on full training set")

Model trained on full training set


In [15]:
final_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', final_model)
])

final_pipeline.fit(X_train, y_train_log)

joblib.dump(final_pipeline, 'uber_fare_pipeline.pkl')
print("Pipeline saved successfully")

Pipeline saved successfully


In [16]:
loaded_pipeline = joblib.load('uber_fare_pipeline.pkl')

sample = X_test.iloc[[0]]
real_fare = y_test.iloc[0]

pred_log = loaded_pipeline.predict(sample)
pred_fare = np.expm1(pred_log)[0]

print("Real fare:", real_fare)
print("Predicted fare:", pred_fare)
print(sample)

Real fare: 7.7
Predicted fare: 8.555722805067445
     Car Condition Weather Traffic Condition  pickup_longitude  \
2484           Bad   windy      Flow Traffic         -1.291231   

      pickup_latitude  dropoff_longitude  dropoff_latitude  passenger_count  \
2484         0.711509          -1.291524          0.711045              1.0   

       day  month  weekday    year   jfk_dist  distance  bearing  hour_sin  \
2484  16.0    3.0      1.0  2010.0  43.818873  3.279181  2.69529 -0.707107   

      hour_cos  
2484  0.707107  
